# 01. Model — 텍스트 생성과 구조화된 답변

이 노트북에서는 [`docs/01.langchain.md`](../docs/01.langchain.md)의 **2장 Model** 내용을 코드로 직접 실습합니다.

다루는 주제는 다음과 같습니다.

1. **`init_chat_model`** 으로 모델 객체 만들기
2. **`stream()`** — 청크(chunk) 단위로 답변을 흘려보내며 받기
3. **`batch()`** — 여러 입력을 동시에 처리하기
4. **Structured Output (Pydantic)** — Pydantic 스키마로 답변을 강제하기
5. **Structured Output (JSON Schema)** — JSON Schema로 답변을 강제하기

> 💡 실행하기 전에 프로젝트 루트에 `.env` 파일이 있고, `OPENAI_API_KEY` 등 필요한 키가 들어있는지 확인하세요.

## 0. 환경 준비 및 모델 선언

LangChain에서는 `init_chat_model` 한 줄로 OpenAI / Anthropic / Google 등 다양한 공급자(provider)의 모델을 동일한 인터페이스로 사용할 수 있습니다.

In [ ]:
# 환경 변수 로더 — .env 파일에서 API 키 등을 읽어온다
from dotenv import load_dotenv

# LangChain의 통합 모델 초기화 함수
from langchain.chat_models import init_chat_model

# 구조화된 답변(Structured Output)에 사용할 Pydantic 모델 도구
from pydantic import BaseModel, Field

In [ ]:
# .env 파일에서 환경 변수(예: OPENAI_API_KEY)를 현재 프로세스로 로드
load_dotenv()

# gpt-4o-mini 모델로 ChatModel 객체 생성
# init_chat_model("provider:model") 형태로 공급자를 명시할 수도 있다.
# 예) init_chat_model("openai:gpt-4o-mini")
model = init_chat_model("gpt-4o-mini")

## 1. Stream — 청크(chunk) 단위로 답변 받기

`invoke()`는 답변이 모두 생성된 뒤에 한 번에 결과를 돌려주지만, **`stream()`** 은 모델이 토큰을 만들어내는 즉시 부분 결과(chunk)를 흘려보냅니다.

- **장점** : ChatGPT처럼 글자가 한 자씩 흐르듯 출력되어 **체감 속도(UX)** 가 좋아집니다.
- **사용법** : `for chunk in model.stream(prompt):` 처럼 반복문으로 청크를 받아 출력합니다.
- 각 청크는 부분 텍스트를 담고 있고, `chunk.text` 또는 `chunk.content` 로 꺼낼 수 있습니다.

In [ ]:
# Stream 호출 — 답변이 생성되는 즉시 청크 단위로 출력
# end='' 로 지정해 줄바꿈 없이 이어붙여 출력한다.
for chunk in model.stream('Explain about the movie The Truman Show, Reply it briefly'):
  print(chunk.text, end='')

## 2. Batch — 여러 입력을 동시에 처리

`batch()`는 여러 개의 프롬프트(입력)를 **한 번에 병렬로** 모델에 전달합니다.

- **장점** : 호출 횟수를 줄여 **응답 시간 단축 + 비용 절감** 효과가 있습니다.
- **사용처** : 대량 데이터 라벨링, 평가셋(eval) 일괄 실행, A/B 비교 등.
- 결과는 입력 순서와 동일한 **응답 객체의 리스트**로 반환됩니다.

In [ ]:
# Batch 호출 — 여러 프롬프트를 리스트로 묶어 한 번에 전달
inputs = [
  'Explain about the movie The Truman Show',
  'Explain about the movie The Truman Show, Reply it briefly'
]

# 결과는 입력 순서와 동일한 AIMessage 리스트로 반환된다.
for response in model.batch(inputs):
  print(response.content)

## 3. Structured Output — Pydantic 스키마로 답변 형태 강제

LLM의 기본 답변은 자연어 문장이지만, 실제 애플리케이션에서는 **JSON / 객체 형태**의 답변이 필요할 때가 많습니다 (예: DB 저장, 다음 단계로 전달).

LangChain에서는 다음 절차로 구현합니다.

1. **Pydantic 모델로 스키마(class)** 를 정의한다.
2. `model.with_structured_output(스키마)` 로 **새로운 모델 객체** 를 만든다.
3. 그 모델을 `invoke()` 하면 **자동으로 스키마에 맞춘 객체** 가 반환된다.

아래 `Movie` 클래스가 그 스키마입니다.

In [ ]:
# 영화 정보를 표현하는 Pydantic 스키마
# - docstring("영화 정보")은 모델 전체에 대한 설명으로 LLM에게 힌트로 전달된다.
# - Field(..., description=...)에서 ...(Ellipsis)는 "이 필드는 필수"라는 뜻이다.
class Movie(BaseModel):
  """영화 정보"""
  title: str = Field(..., description="영화 제목")
  director: str = Field(..., description="감독")
  year: int = Field(..., description="개봉 연도")
  genre: str = Field(..., description="장르")

### 📌 보충 — `Field(..., description="...")` 의 `...`은 무슨 뜻일까?

`Field(..., description="영화 제목")` 에서 `...` (Ellipsis, 생략 부호)는 **Pydantic** 라이브러리에서 **"이 필드는 필수(Required) 값이다"** 라는 것을 명시하는 문법입니다.

#### 1) 필수 필드 지정 (Required Field)

Pydantic 모델을 생성할 때 해당 필드에 기본값을 주지 않고, 반드시 사용자가 값을 입력해야 한다는 뜻입니다.

- **`...` 을 사용하는 경우 (필수)**
    ```python
    title: str = Field(..., description="영화 제목")
    # 객체 생성 시 title을 안 넣으면 에러가 난다.
    # movie = Movie() -> ValidationError!
    ```
- **기본값을 사용하는 경우 (선택)**
    ```python
    title: str = Field("제목 없음", description="영화 제목")
    # title을 안 넣으면 "제목 없음"이 자동으로 들어간다.
    ```

#### 2) 왜 `None` 대신 `...`을 쓰나요?

파이썬에서 `title: str = None` 이라고 쓰면 "기본값이 None이다"라는 뜻이 되어버립니다. 하지만 **"기본값은 없지만, 설명(description) 같은 메타데이터는 추가하고 싶을 때"** `Field` 함수의 첫 번째 인자로 `...` 을 넣어 *"이건 필수값이야!"* 라고 알려주는 것입니다.

#### 3) LangChain / LLM 에서의 역할

LangChain에서 LLM이 특정 구조로 답변을 생성하게 할 때(Structured Output), 이 `...` 이 붙은 필드는 LLM에게 **"이 정보는 반드시 추출하거나 생성해야 해"** 라고 강제하는 역할을 합니다.

#### 요약

- `...` = **"이 값은 필수야, 꼭 넣어줘!"**
- `...` 자리에 다른 값을 넣으면 그 값이 **기본값** 이 됩니다.

In [ ]:
# Movie 스키마를 따르도록 모델을 래핑한다.
# 이 시점부터 model_with_structured_output.invoke(...)는 Movie 객체를 반환한다.
model_with_structured_output = model.with_structured_output(Movie)

# 자연어로 질문해도 결과는 Movie 인스턴스 형태로 돌아온다.
response = model_with_structured_output.invoke("Explain about the movie Truman Show")
print(response)
# 예: title='The Truman Show' director='Peter Weir' year=1998 genre='Psychological comedy-drama'

## 4. Structured Output — JSON Schema 사용

Pydantic 클래스가 아니라 **JSON Schema(딕셔너리)** 로도 출력 형태를 강제할 수 있습니다.

| 방식         | 장점                                               | 단점                                  |
| :----------- | :------------------------------------------------- | :------------------------------------ |
| **Pydantic** | 파싱 + **데이터 검증** 까지 지원, 파이썬 객체로 사용 | 파이썬 클래스 정의가 필요             |
| **JSON Schema** | 언어 중립적, 다른 시스템과 스키마 공유에 유리      | **파싱만 지원**, 검증은 별도로 필요    |

> 💡 LangChain 단독 프로젝트에서는 보통 **Pydantic** 이 더 편리하지만, 외부 시스템과 스키마를 공유해야 한다면 JSON Schema가 유용합니다.

In [ ]:
# 영화 정보 스키마를 JSON Schema(딕셔너리)로 정의
# - "type": "object" → 결과는 객체(딕셔너리) 형태
# - "properties" → 각 필드의 타입과 설명
# - "required" → 반드시 포함되어야 하는 필드 목록
movie_json_schema = {
  "title": "Movie",
  "type": "object",
  "properties": {
    "title":    {"type": "string",  "description": "영화 제목"},
    "director": {"type": "string",  "description": "감독"},
    "year":     {"type": "integer", "description": "개봉 연도"},
    "genre":    {"type": "string",  "description": "장르"}
  },
  "required": ["title", "director", "year", "genre"]
}

# JSON Schema도 Pydantic과 동일하게 with_structured_output에 넘길 수 있다.
# 다만 결과는 Pydantic 객체가 아닌 dict로 반환된다.
model_with_json_schema = model.with_structured_output(movie_json_schema)
response = model_with_json_schema.invoke("Explain about the movie Truman Show")
print(response)
# 예: {'title': 'The Truman Show', 'director': 'Peter Weir', 'year': 1998, 'genre': 'Drama, Sci-Fi'}